In [1]:
pip install requests geopandas


Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import requests
# Optional: import geopandas as gpd  # Uncomment if exporting to Shapefile

ENDPOINTS = {
    "montgomery_county_zipcodes": {
        "url": "https://montgomerycountymd.gov",
        "where": "1=1",
        "out_geojson": "montgomery_county_zipcodes.geojson",
        "out_shp": "montgomery_county_zipcodes.shp",
    },
    "maryland_zcta_2020": {
        "url": "https://md.gov",
        "where": "1=1",
        "out_geojson": "maryland_zcta_2020.geojson",
        "out_shp": "maryland_zcta_2020.shp",
    },
    "maryland_5digit_zipcodes": {
        "url": "https://md.gov",
        "where": "1=1",
        "out_geojson": "maryland_5digit_zipcodes.geojson",
        "out_shp": "maryland_5digit_zipcodes.shp",
    },
}


def fetch_arcgis_geojson(query_url: str, where_clause: str = "1=1", batch_size: int = 1000) -> dict:
    """Fetches all features from an ArcGIS REST endpoint using offset pagination."""
    all_features = []
    offset = 0

    while True:
        params = {
            "where": where_clause,
            "outFields": "*",
            "outSR": "4326",  # Force WGS84 (Lat/Lon) standard GeoJSON CRS
            "f": "geojson",
            "resultOffset": offset,
            "resultRecordCount": batch_size,
        }

        response = requests.get(query_url, params=params, timeout=60)
        response.raise_for_status()
        data = response.json()

        if "error" in data:
            raise RuntimeError(f"ArcGIS API Error: {data['error']}")

        features = data.get("features", [])
        all_features.extend(features)
        print(f"  Fetched {len(features)} features (total: {len(all_features)})...")

        # Stop if fewer records returned than requested or exceededTransferLimit is False
        if len(features) < batch_size and not data.get("exceededTransferLimit", False):
            break

        offset += batch_size

    return {
        "type": "FeatureCollection",
        "features": all_features,
    }


def main():
    for name, cfg in ENDPOINTS.items():
        print(f"\nDownloading [{name}]...")
        try:
            geojson_data = fetch_arcgis_geojson(cfg["url"], cfg["where"])

            with open(cfg["out_geojson"], "w", encoding="utf-8") as f:
                json.dump(geojson_data, f)
            print(f"  Saved GeoJSON -> {cfg['out_geojson']}")

            # Optional: Convert GeoJSON to ESRI Shapefile using GeoPandas
            # gdf = gpd.read_file(cfg["out_geojson"])
            # gdf.to_file(cfg["out_shp"], driver="ESRI Shapefile")
            # print(f"  Saved Shapefile -> {cfg['out_shp']}")

        except Exception as e:
            print(f"  Failed to download {name}: {e}")


if __name__ == "__main__":
    main()



  Failed to download montgomery_county_zipcodes: Expecting value: line 1 column 1 (char 0)

  Failed to download maryland_zcta_2020: Expecting value: line 1 column 1 (char 0)

  Failed to download maryland_5digit_zipcodes: Expecting value: line 1 column 1 (char 0)
